In [ ]:
import os
import cv2 as cv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from skimage.feature import graycomatrix, graycoprops
from scipy.stats import entropy
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay)
import seaborn as sns

Pada tahap ini dilakukan impor berbagai library yang diperlukan untuk proses pengolahan citra, ekstraksi fitur, klasifikasi, dan evaluasi model. Library os digunakan untuk mengakses serta mengelola file dan folder dataset, sedangkan OpenCV (cv2) digunakan untuk membaca, mengubah, dan memproses citra digital. NumPy (np) berfungsi untuk mengolah data dalam bentuk array dan melakukan perhitungan numerik secara efisien, sementara Matplotlib (plt) digunakan untuk menampilkan citra maupun grafik hasil pengolahan. Pandas (pd) digunakan untuk menyimpan dan mengelola data hasil ekstraksi fitur dalam bentuk tabel. Library scikit-learn menyediakan berbagai fungsi untuk klasifikasi, evaluasi model, dan pembagian data, sedangkan skimage.feature digunakan untuk menghitung matriks GLCM dan fitur teksturnya. scipy.stats digunakan untuk menghitung nilai entropy, dan seaborn digunakan untuk visualisasi heatmap korelasi antar fitur.

In [ ]:
data = []
labels = []
file_name = []

IMG_SIZE = (128, 128)

for sub_folder in os.listdir("Assets\\"):
    sub_folder_files = os.listdir(os.path.join("Assets\\", sub_folder))
    for i, filename in enumerate(sub_folder_files):
        img_path = os.path.join("Assets\\", sub_folder, filename)
        img = cv.imread(img_path)
        
        if img is None:
            continue

        data.append(img)
        labels.append(sub_folder)
        file_name.append(f"{sub_folder}_{i+1}.jpg")

print(f"Total data: {len(data)}")
print(f"Kelas: {sorted(set(labels))}")

Kode ini digunakan untuk membaca seluruh citra yang terdapat pada folder dataset dan mempersiapkannya untuk proses pengolahan selanjutnya. Program terlebih dahulu membuat tiga list, yaitu data untuk menyimpan citra mentah, labels untuk menyimpan kelas citra berdasarkan nama subfolder, dan file_name untuk menyimpan nama file gambar. Selanjutnya program melakukan perulangan pada setiap subfolder di dalam folder Assets. Setiap subfolder dianggap sebagai sebuah kelas data. Setiap gambar dibaca menggunakan cv.imread(), kemudian dilakukan pengecekan untuk memastikan gambar berhasil dimuat. Pada tahap ini citra belum diproses, melainkan disimpan dalam bentuk aslinya agar preprocessing dapat dilakukan secara terpisah dan terstruktur.

# Data Preparation

### Define Preprocessing Function

In [ ]:
TARGET_SIZE = (128, 128)

def resize_grayscale(image, target_size=TARGET_SIZE):
    resized = cv.resize(image, target_size)

    if len(resized.shape) == 3:
        gray = cv.cvtColor(resized, cv.COLOR_BGR2GRAY)
    else:
        gray = resized

    return gray.astype(np.uint8)

In [ ]:
def prepro(image):
    # Resize + Grayscale
    img = resize_grayscale(image)
    return img

Kode ini berisi fungsi yang digunakan pada tahap preprocessing citra. Fungsi resize_grayscale() digunakan untuk mengubah ukuran gambar menjadi 128 × 128 piksel agar seluruh citra memiliki dimensi yang seragam. Setelah itu dilakukan pengecekan jumlah kanal citra. Jika citra masih berwarna (memiliki 3 kanal RGB), citra akan dikonversi menjadi grayscale sehingga setiap piksel hanya memiliki satu nilai intensitas. Hasil akhirnya dikembalikan dalam format uint8.

Fungsi prepro() merupakan fungsi utama preprocessing yang memanggil resize_grayscale(). Pada Percobaan 1 ini, preprocessing yang diterapkan hanya dua tahap yaitu Resize dan Grayscale, tanpa penambahan filter tambahan. Pendekatan ini digunakan sebagai baseline untuk melihat kemampuan fitur GLCM dalam mengklasifikasikan jenis batuan dengan preprocessing paling minimal.

## Percobaan 1 (Resize + Grayscale)

Kode ini digunakan untuk menerapkan proses preprocessing pada seluruh citra yang terdapat dalam dataset serta menampilkan hasilnya berdasarkan kelas masing-masing. Fungsi percobaan1() memanggil fungsi prepro() yang berisi rangkaian tahapan preprocessing, kemudian mengembalikan citra hasil pengolahan. Selanjutnya, seluruh citra pada variabel data diproses menggunakan list comprehension dan disimpan ke dalam variabel dataPreprocessed.

Program kemudian mengambil seluruh label unik yang terdapat pada dataset dan melakukan perulangan untuk setiap kelas. Pada setiap iterasi, program mengambil indeks gambar yang sesuai dengan kelas tersebut, lalu menampilkan hingga 24 citra hasil preprocessing dalam bentuk grid 4 baris × 6 kolom menggunakan Matplotlib.

In [ ]:
def percobaan1(img):
    hasil = prepro(img)
    return hasil

dataPreprocessed = [percobaan1(img) for img in data]

unique_labels = sorted(set(labels))

for label in unique_labels:
    idxs = [j for j, l in enumerate(labels) if l == label]
    
    fig, axs = plt.subplots(4, 6, figsize=(12, 8))
    fig.suptitle(f'{label}', fontsize=16)
    
    for k in range(min(24, len(idxs))):
        row = k // 6
        col = k % 6
        ax = axs[row][col]
        
        ax.imshow(dataPreprocessed[idxs[k]], cmap='gray')
        ax.axis('off')
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

In [ ]:
def glcm(image, derajat):
    if derajat == 0:
        angles = [0]
    elif derajat == 45:
        angles = [np.pi / 4]
    elif derajat == 90:
        angles = [np.pi / 2]
    elif derajat == 135:
        angles = [3 * np.pi / 4]
    else:
        raise ValueError("Invalid angle. It should be one of the following: 0, 45, 90, 135.")
    
    glcm = graycomatrix(image, [1], angles, 256, symmetric=True, normed=True)
    return glcm

Kode ini digunakan untuk membentuk Gray Level Co-occurrence Matrix (GLCM) dari sebuah citra. Fungsi glcm() menerima dua parameter, yaitu image yang merupakan citra masukan dan derajat yang menunjukkan arah hubungan antar piksel yang akan dianalisis. Program terlebih dahulu mengubah nilai sudut yang diberikan, yaitu 0°, 45°, 90°, atau 135°, ke dalam satuan radian karena fungsi graycomatrix() menggunakan radian sebagai parameter sudut. Jika nilai sudut yang dimasukkan tidak sesuai dengan pilihan yang tersedia, program akan menampilkan pesan kesalahan.

Setelah sudut ditentukan, fungsi graycomatrix() dipanggil dengan parameter distance bernilai 1, artinya hubungan antar piksel dihitung berdasarkan piksel yang langsung bersebelahan. Parameter symmetric=True digunakan agar matriks bersifat simetris, dan normed=True digunakan agar nilai matriks dinormalisasi sehingga jumlah seluruh elemennya bernilai 1.

In [ ]:
def correlation_feat(matriks):
    return graycoprops(matriks, 'correlation')[0, 0]

def dissimilarity(matriks):
    return graycoprops(matriks, 'dissimilarity')[0, 0]

def homogenity(matriks):
    return graycoprops(matriks, 'homogeneity')[0, 0]

def contrast(matriks):
    return graycoprops(matriks, 'contrast')[0, 0]

def ASM(matriks):
    return graycoprops(matriks, 'ASM')[0, 0]

def energy(matriks):
    return graycoprops(matriks, 'energy')[0, 0]

def entropyGlcm(matriks):
    return entropy(matriks.ravel())

Kode ini digunakan untuk menghitung berbagai fitur tekstur dari matriks Gray Level Co-occurrence Matrix (GLCM). Setiap fungsi menerima parameter berupa matriks GLCM yang telah dibentuk sebelumnya, kemudian menghitung karakteristik tekstur tertentu menggunakan fungsi graycoprops() dari pustaka scikit-image. Fitur correlation digunakan untuk mengukur hubungan atau ketergantungan antar nilai piksel yang berdekatan. Fitur dissimilarity mengukur tingkat perbedaan nilai keabuan antar pasangan piksel, sedangkan homogeneity mengukur tingkat keseragaman tekstur pada citra. Fitur contrast digunakan untuk mengukur perbedaan intensitas antara piksel dan tetangganya, ASM (Angular Second Moment) mengukur keseragaman distribusi nilai intensitas, dan energy merupakan akar kuadrat dari ASM. Fitur entropy dihitung secara manual menggunakan fungsi dari scipy.stats untuk mengukur tingkat ketidakaturan atau kompleksitas tekstur citra.

In [ ]:
Derajat0 = []
Derajat45 = []
Derajat90 = []
Derajat135 = []

for i in range(len(dataPreprocessed)):
    D0   = glcm(dataPreprocessed[i], 0)
    D45  = glcm(dataPreprocessed[i], 45)
    D90  = glcm(dataPreprocessed[i], 90)
    D135 = glcm(dataPreprocessed[i], 135)
    Derajat0.append(D0)
    Derajat45.append(D45)
    Derajat90.append(D90)
    Derajat135.append(D135)

Kontras0, Kontras45, Kontras90, Kontras135         = [], [], [], []
dissimilarity0, dissimilarity45, dissimilarity90, dissimilarity135 = [], [], [], []
homogenity0, homogenity45, homogenity90, homogenity135 = [], [], [], []
entropy0, entropy45, entropy90, entropy135         = [], [], [], []
ASM0, ASM45, ASM90, ASM135                         = [], [], [], []
energy0, energy45, energy90, energy135             = [], [], [], []
correlation0, correlation45, correlation90, correlation135 = [], [], [], []

for i in range(len(dataPreprocessed)):
    Kontras0.append(contrast(Derajat0[i]))
    Kontras45.append(contrast(Derajat45[i]))
    Kontras90.append(contrast(Derajat90[i]))
    Kontras135.append(contrast(Derajat135[i]))

    dissimilarity0.append(dissimilarity(Derajat0[i]))
    dissimilarity45.append(dissimilarity(Derajat45[i]))
    dissimilarity90.append(dissimilarity(Derajat90[i]))
    dissimilarity135.append(dissimilarity(Derajat135[i]))

    homogenity0.append(homogenity(Derajat0[i]))
    homogenity45.append(homogenity(Derajat45[i]))
    homogenity90.append(homogenity(Derajat90[i]))
    homogenity135.append(homogenity(Derajat135[i]))

    entropy0.append(entropyGlcm(Derajat0[i]))
    entropy45.append(entropyGlcm(Derajat45[i]))
    entropy90.append(entropyGlcm(Derajat90[i]))
    entropy135.append(entropyGlcm(Derajat135[i]))

    ASM0.append(ASM(Derajat0[i]))
    ASM45.append(ASM(Derajat45[i]))
    ASM90.append(ASM(Derajat90[i]))
    ASM135.append(ASM(Derajat135[i]))

    energy0.append(energy(Derajat0[i]))
    energy45.append(energy(Derajat45[i]))
    energy90.append(energy(Derajat90[i]))
    energy135.append(energy(Derajat135[i]))

    correlation0.append(correlation_feat(Derajat0[i]))
    correlation45.append(correlation_feat(Derajat45[i]))
    correlation90.append(correlation_feat(Derajat90[i]))
    correlation135.append(correlation_feat(Derajat135[i]))

print(f"Ekstraksi fitur selesai untuk {len(dataPreprocessed)} citra.")

Kode ini digunakan untuk melakukan ekstraksi fitur tekstur GLCM pada seluruh citra hasil preprocessing. Pertama, program membuat empat list kosong untuk menyimpan matriks GLCM berdasarkan arah sudut 0°, 45°, 90°, dan 135°. Setiap citra pada dataPreprocessed kemudian diproses menggunakan fungsi glcm() pada keempat sudut tersebut, lalu hasil matriksnya disimpan ke dalam list sesuai arah sudutnya.

Setelah matriks GLCM terbentuk, program membuat beberapa list kosong untuk menyimpan nilai fitur tekstur dari masing-masing sudut. Fitur yang dihitung meliputi correlation, contrast, dissimilarity, homogeneity, entropy, ASM, dan energy. Setiap fitur dihitung untuk keempat arah sudut sehingga menghasilkan 28 fitur secara keseluruhan per citra.

## Ekstraksi ke CSV
Kode ini digunakan untuk menggabungkan seluruh hasil ekstraksi fitur GLCM ke dalam sebuah tabel yang terstruktur. Program membuat sebuah dictionary bernama dataTable yang berisi nama file, label kelas, serta seluruh fitur tekstur yang telah dihitung sebelumnya, yaitu contrast, homogeneity, dissimilarity, entropy, ASM, energy, dan correlation pada sudut 0°, 45°, 90°, dan 135°. Setiap kolom pada dictionary merepresentasikan satu jenis fitur, sedangkan setiap baris merepresentasikan satu citra.
Selanjutnya dictionary tersebut dikonversi menjadi DataFrame menggunakan Pandas dan disimpan ke dalam file CSV bernama hasil_ekstraksi_Percobaan1.csv agar dapat digunakan kembali tanpa perlu menghitung ulang fitur dari awal.

In [ ]:
dataTable = {
    'Filename': file_name, 'Label': labels,
    'Contrast0': Kontras0, 'Contrast45': Kontras45, 'Contrast90': Kontras90, 'Contrast135': Kontras135,
    'Homogeneity0': homogenity0, 'Homogeneity45': homogenity45, 'Homogeneity90': homogenity90, 'Homogeneity135': homogenity135,
    'Dissimilarity0': dissimilarity0, 'Dissimilarity45': dissimilarity45, 'Dissimilarity90': dissimilarity90, 'Dissimilarity135': dissimilarity135,
    'Entropy0': entropy0, 'Entropy45': entropy45, 'Entropy90': entropy90, 'Entropy135': entropy135,
    'ASM0': ASM0, 'ASM45': ASM45, 'ASM90': ASM90, 'ASM135': ASM135,
    'Energy0': energy0, 'Energy45': energy45, 'Energy90': energy90, 'Energy135': energy135,
    'Correlation0': correlation0, 'Correlation45': correlation45, 'Correlation90': correlation90, 'Correlation135': correlation135,
}
df = pd.DataFrame(dataTable)
df.to_csv('hasil_ekstraksi_Percobaan1.csv', index=False)

hasilEkstrak = pd.read_csv('hasil_ekstraksi_Percobaan1.csv')
hasilEkstrak

## Feature Selection
Seleksi fitur dilakukan menggunakan metode korelasi antar fitur dengan threshold 0.95. Fitur yang memiliki nilai korelasi absolut lebih dari atau sama dengan threshold akan dihapus karena dianggap membawa informasi yang sama dengan fitur lain sehingga bersifat redundan.

Proses seleksi dimulai dengan menghitung matriks korelasi dari seluruh fitur hasil ekstraksi. Kemudian dibuat array boolean yang menandai fitur mana yang akan dipertahankan. Jika dua fitur memiliki korelasi absolut ≥ 0.95, maka fitur kedua (j) akan ditandai untuk dihapus. Fitur-fitur yang lolos seleksi kemudian digunakan sebagai input model klasifikasi.

In [ ]:
corr_matrix = hasilEkstrak.drop(columns=['Label','Filename']).corr()

threshold = 0.95
columns = np.full((corr_matrix.shape[0],), True, dtype=bool)
for i in range(corr_matrix.shape[0]):
    for j in range(i+1, corr_matrix.shape[0]):
        if abs(corr_matrix.iloc[i,j]) >= threshold:
            if columns[j]:
                columns[j] = False

select = hasilEkstrak.drop(columns=['Label','Filename']).columns[columns]
x_new = hasilEkstrak[select]
y = hasilEkstrak['Label']

print(f"Fitur sebelum seleksi : 28")
print(f"Fitur setelah seleksi : {len(select)}")
print(f"Fitur terpilih        : {list(select)}")

plt.figure(figsize=(17,17))
sns.heatmap(x_new.corr(), annot=True, cmap='Blues', fmt=".2f")
plt.title('Heatmap Korelasi Fitur - Percobaan 1', fontsize=14)
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x_new, y, test_size=0.2, random_state=42)
print(X_train.shape)
print(X_test.shape)

## Splitting Data
Data dibagi menjadi dua bagian yaitu data training dan data testing menggunakan perbandingan 80:20, artinya 80% data digunakan untuk melatih model dan 20% sisanya digunakan untuk menguji performa model. Pembagian ini dipilih karena jumlah data yang tersedia tidak terlalu banyak, sehingga porsi training yang lebih besar dapat membantu model belajar lebih optimal. Pembagian dilakukan menggunakan fungsi train_test_split() dengan random_state=42 agar hasil pembagian konsisten dan dapat direproduksi.

In [ ]:
mean_train = X_train.mean()
std_train  = X_train.std()

X_test  = (X_test  - mean_train) / std_train
X_train = (X_train - mean_train) / std_train

## Normalisasi Fitur
Normalisasi dilakukan menggunakan metode Z-score (Standardization), yaitu dengan mengurangi nilai rata-rata lalu dibagi dengan standar deviasi. Proses normalisasi menggunakan parameter mean dan standar deviasi dari data training, dan parameter yang sama diterapkan juga ke data testing. Hal ini dilakukan agar tidak terjadi data leakage, yaitu kondisi di mana informasi dari data testing ikut mempengaruhi proses training. Setelah normalisasi, setiap fitur akan memiliki distribusi dengan rata-rata mendekati 0 dan standar deviasi mendekati 1, sehingga seluruh fitur berada dalam skala yang seragam dan model tidak terpengaruh oleh perbedaan skala antar fitur.

# MODELING

In [ ]:
def generateClassificationReport(y_true, y_pred):
    print(classification_report(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))
    print('Accuracy:', accuracy_score(y_true, y_pred))

# Define classifiers
rf  = RandomForestClassifier(n_estimators=5, random_state=42)
svm = SVC(kernel='rbf', random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)

## Define Model
Kode ini digunakan untuk menyiapkan proses evaluasi dan klasifikasi data. Fungsi generateClassificationReport() dibuat untuk menampilkan hasil evaluasi model klasifikasi. Fungsi ini menerima data label sebenarnya (y_true) dan hasil prediksi model (y_pred), kemudian menampilkan classification report yang berisi nilai precision, recall, f1-score, dan support untuk setiap kelas. Selain itu, fungsi juga menampilkan confusion matrix untuk melihat jumlah prediksi yang benar maupun salah pada masing-masing kelas, serta nilai accuracy untuk mengetahui tingkat ketepatan model secara keseluruhan.

Tiga model klasifikasi yang digunakan adalah Random Forest dengan 5 pohon keputusan, Support Vector Machine (SVM) dengan kernel RBF, dan K-Nearest Neighbors (KNN) dengan nilai k=5.

## Hasil Modeling Random Forest
* ### Training Set
Isi analisis hasil training Random Forest pada Percobaan 1 di sini setelah menjalankan notebook.
* ### Testing Set
Isi analisis hasil testing Random Forest pada Percobaan 1 di sini setelah menjalankan notebook.

In [ ]:
# Train Random Forest Classifier
rf.fit(X_train, y_train)

# Make predictions and evaluate the model with the training set
print("------Training Set------")
y_pred = rf.predict(X_train)
generateClassificationReport(y_train, y_pred)

# Make predictions and evaluate the model with the testing set
print("\n------Testing Set------")
y_pred = rf.predict(X_test)
generateClassificationReport(y_test, y_pred)

## Hasil Modeling SVM
* ### Training Set
Isi analisis hasil training SVM pada Percobaan 1 di sini setelah menjalankan notebook.
* ### Testing Set
Isi analisis hasil testing SVM pada Percobaan 1 di sini setelah menjalankan notebook.

In [ ]:
# Train SVM Classifier
svm.fit(X_train, y_train)

# Make predictions and evaluate the model with the training set
print("\n------Training Set------")
y_pred = svm.predict(X_train)
generateClassificationReport(y_train, y_pred)

# Make predictions and evaluate the model with the testing set
print("\n------Testing Set------")
y_pred = svm.predict(X_test)
generateClassificationReport(y_test, y_pred)

## Hasil Modeling KNN
* ### Training Set
Isi analisis hasil training KNN pada Percobaan 1 di sini setelah menjalankan notebook.
* ### Testing Set
Isi analisis hasil testing KNN pada Percobaan 1 di sini setelah menjalankan notebook.

In [ ]:
# Train KNN Classifier
knn.fit(X_train, y_train)

# Make predictions and evaluate the model with the training set
print("\n------Training Set------")
y_pred = knn.predict(X_train)
generateClassificationReport(y_train, y_pred)

# Make predictions and evaluate the model with the testing set
print("\n------Testing Set------")
y_pred = knn.predict(X_test)
generateClassificationReport(y_test, y_pred)

## Evaluasi Confusion Matrix
Confusion matrix digunakan untuk melihat secara detail bagaimana model melakukan prediksi pada setiap kelas. Sumbu vertikal menunjukkan label sebenarnya (true label) dan sumbu horizontal menunjukkan label yang diprediksi (predicted label). Angka 0 merepresentasikan Coal, 1 merepresentasikan Limestone, dan 2 merepresentasikan Sandstone.

Isi analisis confusion matrix dari ketiga model pada Percobaan 1 di sini setelah menjalankan notebook.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm   = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(cmap=plt.cm.Blues)
    plt.title(title)
    plt.show()

# Plot confusion matrix for Random Forest
plot_confusion_matrix(y_test, rf.predict(X_test),  "Random Forest Confusion Matrix - Percobaan 1")
# Plot confusion matrix for SVM
plot_confusion_matrix(y_test, svm.predict(X_test), "SVM Confusion Matrix - Percobaan 1")
# Plot confusion matrix for KNN
plot_confusion_matrix(y_test, knn.predict(X_test), "KNN Confusion Matrix - Percobaan 1")

## Simpan Hasil Klasifikasi ke CSV

In [ ]:
# Simpan hasil perbandingan model
hasil_klasifikasi = {
    'Model'    : ['Random Forest', 'SVM', 'KNN'],
    'Accuracy_Train': [
        accuracy_score(y_train, rf.predict(X_train)),
        accuracy_score(y_train, svm.predict(X_train)),
        accuracy_score(y_train, knn.predict(X_train)),
    ],
    'Accuracy_Test': [
        accuracy_score(y_test, rf.predict(X_test)),
        accuracy_score(y_test, svm.predict(X_test)),
        accuracy_score(y_test, knn.predict(X_test)),
    ],
    'Precision': [
        precision_score(y_test, rf.predict(X_test),  average='weighted', zero_division=0),
        precision_score(y_test, svm.predict(X_test), average='weighted', zero_division=0),
        precision_score(y_test, knn.predict(X_test), average='weighted', zero_division=0),
    ],
    'Recall': [
        recall_score(y_test, rf.predict(X_test),  average='weighted', zero_division=0),
        recall_score(y_test, svm.predict(X_test), average='weighted', zero_division=0),
        recall_score(y_test, knn.predict(X_test), average='weighted', zero_division=0),
    ],
    'F1_Score': [
        f1_score(y_test, rf.predict(X_test),  average='weighted', zero_division=0),
        f1_score(y_test, svm.predict(X_test), average='weighted', zero_division=0),
        f1_score(y_test, knn.predict(X_test), average='weighted', zero_division=0),
    ],
}
df_hasil = pd.DataFrame(hasil_klasifikasi)
df_hasil.to_csv('hasil_klasifikasi_Percobaan1.csv', index=False)
print("✅ File hasil_klasifikasi_Percobaan1.csv berhasil disimpan!")
df_hasil